# Polars Fundamentos

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/17_polars/code/01_polars_fundamentos.ipynb)

Este notebook acompaña `03_polars_puro.md` y cubre los fundamentos de Polars de forma práctica. Seguimos el patrón **predecir → medir → explicar**: antes de cada bloque de código, formula tu predicción; después, compara con el resultado real y lee la explicación.

> **Referencia:** Sección 17.3 en `03_polars_puro.md` explica la filosofía de expresiones y contextos que aquí pondremos en práctica.

**Tiempo estimado**: ~45 minutos

In [1]:
%pip install polars==1.27.0

Defaulting to user installation because normal site-packages is not writeable


Note: you may need to restart the kernel to use updated packages.


In [2]:
import polars as pl
import time
from datetime import date, datetime
import random

# Semilla fija para reproducibilidad
random.seed(42)

print(f"polars: {pl.__version__}")

polars: 1.27.0


---
## §1: Series y DataFrame — creación e inspección

Un `DataFrame` en Polars es una colección de columnas (`Series`) con nombre y tipo estricto. A diferencia de pandas, Polars **no tiene índice** — las filas se identifican por posición, no por etiqueta.

### Predecir

Antes de ejecutar la siguiente celda, responde mentalmente:

> **¿Qué tipo inferirá Polars para una columna con `[1, 2, None, 4]`?**
>
> Pistas:
> - En pandas, una columna con enteros y `None` se convierte a `float64` (porque `NaN` es float).
> - Polars usa Apache Arrow, que tiene nulls nativos separados del tipo.

Vamos a crear un DataFrame con varios tipos de datos y ver cómo Polars los infiere.

In [3]:
# Crear un DataFrame con múltiples tipos
df = pl.DataFrame({
    "nombre": ["Alice", "Bob", "Carol", "David", "Eva"],
    "edad": [30, 25, 35, 28, None],             # entero con un null
    "salario": [50000.0, 42000.5, 68000.0, 55000.0, 47000.0],  # flotante
    "activo": [True, False, True, True, False],  # booleano
    "fecha_ingreso": [
        date(2020, 1, 15),
        date(2021, 6, 1),
        date(2019, 3, 20),
        date(2022, 11, 10),
        date(2023, 7, 5),
    ],
    "ciudad": ["Madrid", "Lima", "Tokyo", "Madrid", "Lima"],
})

print(df)

shape: (5, 6)
┌────────┬──────┬─────────┬────────┬───────────────┬────────┐
│ nombre ┆ edad ┆ salario ┆ activo ┆ fecha_ingreso ┆ ciudad │
│ ---    ┆ ---  ┆ ---     ┆ ---    ┆ ---           ┆ ---    │
│ str    ┆ i64  ┆ f64     ┆ bool   ┆ date          ┆ str    │
╞════════╪══════╪═════════╪════════╪═══════════════╪════════╡
│ Alice  ┆ 30   ┆ 50000.0 ┆ true   ┆ 2020-01-15    ┆ Madrid │
│ Bob    ┆ 25   ┆ 42000.5 ┆ false  ┆ 2021-06-01    ┆ Lima   │
│ Carol  ┆ 35   ┆ 68000.0 ┆ true   ┆ 2019-03-20    ┆ Tokyo  │
│ David  ┆ 28   ┆ 55000.0 ┆ true   ┆ 2022-11-10    ┆ Madrid │
│ Eva    ┆ null ┆ 47000.0 ┆ false  ┆ 2023-07-05    ┆ Lima   │
└────────┴──────┴─────────┴────────┴───────────────┴────────┘


In [4]:
# Inspección: schema muestra el diccionario nombre → tipo
print("Schema:")
print(df.schema)
print()

# dtypes es una lista de tipos en orden de columna
print("Dtypes:")
print(df.dtypes)
print()

# describe() da estadísticas descriptivas
print("Describe:")
print(df.describe())
print()

# glimpse() transpone la vista — útil cuando hay muchas columnas
print("Glimpse:")
df.glimpse()

Schema:
Schema([('nombre', String), ('edad', Int64), ('salario', Float64), ('activo', Boolean), ('fecha_ingreso', Date), ('ciudad', String)])

Dtypes:
[String, Int64, Float64, Boolean, Date, String]

Describe:
shape: (9, 7)
┌────────────┬────────┬──────────┬─────────────┬────────┬─────────────────────┬────────┐
│ statistic  ┆ nombre ┆ edad     ┆ salario     ┆ activo ┆ fecha_ingreso       ┆ ciudad │
│ ---        ┆ ---    ┆ ---      ┆ ---         ┆ ---    ┆ ---                 ┆ ---    │
│ str        ┆ str    ┆ f64      ┆ f64         ┆ f64    ┆ str                 ┆ str    │
╞════════════╪════════╪══════════╪═════════════╪════════╪═════════════════════╪════════╡
│ count      ┆ 5      ┆ 4.0      ┆ 5.0         ┆ 5.0    ┆ 5                   ┆ 5      │
│ null_count ┆ 0      ┆ 1.0      ┆ 0.0         ┆ 0.0    ┆ 0                   ┆ 0      │
│ mean       ┆ null   ┆ 29.5     ┆ 52400.1     ┆ 0.6    ┆ 2021-05-28 19:12:00 ┆ null   │
│ std        ┆ null   ┆ 4.203173 ┆ 9914.504529 ┆ null   ┆ null  

In [5]:
# Comparación explícita: tipo de columna con null
# En pandas, [1, 2, None, 4] se convierte a float64
# En Polars, se mantiene como Int64 con un null nativo

col_con_null = pl.Series("prueba", [1, 2, None, 4])
print(f"Tipo en Polars: {col_con_null.dtype}")   # Int64, NO Float64
print(f"Null count: {col_con_null.null_count()}")
print(col_con_null)

Tipo en Polars: Int64
Null count: 1
shape: (4,)
Series: 'prueba' [i64]
[
	1
	2
	null
	4
]


### Explicación

**Respuesta a la predicción:** Polars infiere `Int64` para `[1, 2, None, 4]`, **no** `Float64`.

Esto es posible porque Polars usa Apache Arrow como backend de memoria. En Arrow, los nulls se representan con un **bitmap de validez** separado de los datos — no necesitan un valor centinela como `NaN`. Esto tiene tres consecuencias importantes:

1. **Preservación de tipo:** Un entero con nulls sigue siendo entero. No hay promoción accidental a float.
2. **Eficiencia de memoria:** Los nulls ocupan 1 bit por fila (en el bitmap), no 8 bytes como un `float64 NaN`.
3. **Semántica consistente:** `is_null()` funciona igual para todos los tipos. En pandas, `isna()` detecta `NaN` en floats pero no siempre `None` en strings o enteros.

> **Referencia:** Sección 17.3 §1 en `03_polars_puro.md` cubre la creación e inspección de DataFrames.

---
## §2: Expresiones — el concepto central

En Polars, una **expresión** es un objeto que **describe** una computación, pero **no la ejecuta**. Es como escribir una receta sin cocinar: defines qué hacer con los datos, y Polars decide cuándo y cómo ejecutarlo.

### Predecir

> **¿Se ejecuta algo cuando escribes `pl.col('x') + 1` sin contexto?**
>
> En pandas, `df['x'] + 1` se ejecuta inmediatamente y retorna una Serie.
> En Polars, `pl.col('x')` no tiene un DataFrame asociado todavía.

Vamos a ver qué tipo de objeto es una expresión y cómo se compone.

In [6]:
# Una expresión es un OBJETO, no un resultado
expr = pl.col("salario") * 1.1
print(f"Tipo: {type(expr)}")
print(f"Repr: {expr}")
print()
# No hay datos, no hay resultado — solo una descripción
# Esto es fundamentalmente diferente a pandas

Tipo: <class 'polars.expr.expr.Expr'>
Repr: [(col("salario")) * (dyn float: 1.1)]



In [7]:
# Expresiones básicas
print("pl.col('edad'):         ", pl.col("edad"))             # referencia a columna
print("pl.col('edad') + 1:     ", pl.col("edad") + 1)        # aritmética
print("pl.lit(42):             ", pl.lit(42))                 # valor literal (constante)
print("pl.col('edad').alias(): ", pl.col("edad").alias("age"))  # renombrar resultado

pl.col('edad'):          col("edad")
pl.col('edad') + 1:      [(col("edad")) + (dyn int: 1)]
pl.lit(42):              dyn int: 42
pl.col('edad').alias():  col("edad").alias("age")


In [8]:
# Composición por encadenamiento:
# Cada método retorna una NUEVA expresión, nunca muta la original

expr_compuesta = (
    pl.col("nombre")
    .str.to_lowercase()
    .str.replace(" ", "_")
    .alias("nombre_limpio")
)
print(f"Tipo: {type(expr_compuesta)}")
print(f"Repr: {expr_compuesta}")
print()

# Solo se ejecuta cuando la ponemos en un contexto:
resultado = df.select(expr_compuesta)
print(resultado)

Tipo: <class 'polars.expr.expr.Expr'>
Repr: col("nombre").str.lowercase().str.replace([" ", "_"]).alias("nombre_limpio")

shape: (5, 1)
┌───────────────┐
│ nombre_limpio │
│ ---           │
│ str           │
╞═══════════════╡
│ alice         │
│ bob           │
│ carol         │
│ david         │
│ eva           │
└───────────────┘


In [9]:
# pl.when / then / otherwise — el equivalente de if/else vectorizado
categorias = (
    pl.when(pl.col("edad") < 28)
    .then(pl.lit("junior"))
    .when(pl.col("edad") < 33)
    .then(pl.lit("mid"))
    .otherwise(pl.lit("senior"))
    .alias("categoria")
)

# Esto sigue siendo una expresión hasta que la ponemos en contexto
print(df.select("nombre", "edad", categorias))

shape: (5, 3)
┌────────┬──────┬───────────┐
│ nombre ┆ edad ┆ categoria │
│ ---    ┆ ---  ┆ ---       │
│ str    ┆ i64  ┆ str       │
╞════════╪══════╪═══════════╡
│ Alice  ┆ 30   ┆ mid       │
│ Bob    ┆ 25   ┆ junior    │
│ Carol  ┆ 35   ┆ senior    │
│ David  ┆ 28   ┆ mid       │
│ Eva    ┆ null ┆ senior    │
└────────┴──────┴───────────┘


### Explicación

**Respuesta a la predicción:** No, `pl.col('x') + 1` **no ejecuta nada**. Retorna un objeto de tipo `polars.Expr` que representa la operación "tomar la columna x y sumarle 1". No hay DataFrame involucrado todavía.

Esto es el corazón del diseño de Polars:

- **Separación entre descripción y ejecución:** Las expresiones describen *qué* hacer. Los contextos (`select`, `filter`, etc.) definen *dónde* ejecutar.
- **Optimización:** Como las expresiones son objetos, Polars puede analizarlas, reordenarlas y fusionarlas antes de ejecutar.
- **Composición segura:** Encadenar métodos crea nuevas expresiones sin efectos secundarios. No hay `SettingWithCopyWarning`.

> **Referencia:** Sección 17.3 §2 en `03_polars_puro.md` explica por qué las expresiones son superiores a la indexación booleana de pandas.

---
## §3: Contextos — select, with_columns, group_by.agg, filter

Una expresión sola no hace nada. Necesita un **contexto** — un método del DataFrame que le dice "ejecuta aquí". Los cuatro contextos principales son:

| Contexto | Qué retorna |
|---|---|
| `select` | SOLO las columnas especificadas |
| `with_columns` | TODAS las columnas + nuevas/modificadas |
| `group_by.agg` | Filas agrupadas con expresiones de agregación |
| `filter` | Filas que cumplen la condición booleana |

Vamos a aplicar los cuatro al mismo DataFrame para ver la diferencia.

In [10]:
# Recordar nuestro DataFrame
print(df)
print(f"\nShape: {df.shape}")

shape: (5, 6)
┌────────┬──────┬─────────┬────────┬───────────────┬────────┐
│ nombre ┆ edad ┆ salario ┆ activo ┆ fecha_ingreso ┆ ciudad │
│ ---    ┆ ---  ┆ ---     ┆ ---    ┆ ---           ┆ ---    │
│ str    ┆ i64  ┆ f64     ┆ bool   ┆ date          ┆ str    │
╞════════╪══════╪═════════╪════════╪═══════════════╪════════╡
│ Alice  ┆ 30   ┆ 50000.0 ┆ true   ┆ 2020-01-15    ┆ Madrid │
│ Bob    ┆ 25   ┆ 42000.5 ┆ false  ┆ 2021-06-01    ┆ Lima   │
│ Carol  ┆ 35   ┆ 68000.0 ┆ true   ┆ 2019-03-20    ┆ Tokyo  │
│ David  ┆ 28   ┆ 55000.0 ┆ true   ┆ 2022-11-10    ┆ Madrid │
│ Eva    ┆ null ┆ 47000.0 ┆ false  ┆ 2023-07-05    ┆ Lima   │
└────────┴──────┴─────────┴────────┴───────────────┴────────┘

Shape: (5, 6)


In [11]:
# CONTEXTO 1: select — retorna SOLO las columnas que pides
# Puedes pasar strings (columnas sin transformar) o expresiones
resultado_select = df.select(
    "nombre",                                     # columna sin cambio
    (pl.col("salario") * 12).alias("salario_anual"),  # expresión
    (pl.col("edad") >= 30).alias("es_senior"),        # expresión booleana
)
print("select — shape:", resultado_select.shape)  # solo 3 columnas
print(resultado_select)

select — shape: (5, 3)
shape: (5, 3)
┌────────┬───────────────┬───────────┐
│ nombre ┆ salario_anual ┆ es_senior │
│ ---    ┆ ---           ┆ ---       │
│ str    ┆ f64           ┆ bool      │
╞════════╪═══════════════╪═══════════╡
│ Alice  ┆ 600000.0      ┆ true      │
│ Bob    ┆ 504006.0      ┆ false     │
│ Carol  ┆ 816000.0      ┆ true      │
│ David  ┆ 660000.0      ┆ false     │
│ Eva    ┆ 564000.0      ┆ null      │
└────────┴───────────────┴───────────┘


In [12]:
# CONTEXTO 2: with_columns — retorna TODAS las columnas + las nuevas
resultado_wc = df.with_columns(
    (pl.col("salario") * 12).alias("salario_anual"),
    pl.col("nombre").str.to_uppercase().alias("nombre_upper"),
)
print("with_columns — shape:", resultado_wc.shape)  # 6 originales + 2 nuevas = 8
print(resultado_wc)

with_columns — shape: (5, 8)
shape: (5, 8)
┌────────┬──────┬─────────┬────────┬───────────────┬────────┬───────────────┬──────────────┐
│ nombre ┆ edad ┆ salario ┆ activo ┆ fecha_ingreso ┆ ciudad ┆ salario_anual ┆ nombre_upper │
│ ---    ┆ ---  ┆ ---     ┆ ---    ┆ ---           ┆ ---    ┆ ---           ┆ ---          │
│ str    ┆ i64  ┆ f64     ┆ bool   ┆ date          ┆ str    ┆ f64           ┆ str          │
╞════════╪══════╪═════════╪════════╪═══════════════╪════════╪═══════════════╪══════════════╡
│ Alice  ┆ 30   ┆ 50000.0 ┆ true   ┆ 2020-01-15    ┆ Madrid ┆ 600000.0      ┆ ALICE        │
│ Bob    ┆ 25   ┆ 42000.5 ┆ false  ┆ 2021-06-01    ┆ Lima   ┆ 504006.0      ┆ BOB          │
│ Carol  ┆ 35   ┆ 68000.0 ┆ true   ┆ 2019-03-20    ┆ Tokyo  ┆ 816000.0      ┆ CAROL        │
│ David  ┆ 28   ┆ 55000.0 ┆ true   ┆ 2022-11-10    ┆ Madrid ┆ 660000.0      ┆ DAVID        │
│ Eva    ┆ null ┆ 47000.0 ┆ false  ┆ 2023-07-05    ┆ Lima   ┆ 564000.0      ┆ EVA          │
└────────┴──────┴─────────┴

In [13]:
# CONTEXTO 3: group_by.agg — agrupar y aplicar expresiones de agregación
# Cada expresión dentro de agg() produce una columna en el resultado
resultado_gb = df.group_by("ciudad").agg(
    pl.col("salario").mean().alias("salario_medio"),
    pl.col("nombre").count().alias("n_personas"),
    pl.col("edad").max().alias("edad_max"),
    # Expresión compuesta dentro de agg:
    (pl.col("salario").max() - pl.col("salario").min()).alias("rango_salario"),
)
print("group_by.agg — shape:", resultado_gb.shape)
print(resultado_gb)

group_by.agg — shape: (3, 5)
shape: (3, 5)
┌────────┬───────────────┬────────────┬──────────┬───────────────┐
│ ciudad ┆ salario_medio ┆ n_personas ┆ edad_max ┆ rango_salario │
│ ---    ┆ ---           ┆ ---        ┆ ---      ┆ ---           │
│ str    ┆ f64           ┆ u32        ┆ i64      ┆ f64           │
╞════════╪═══════════════╪════════════╪══════════╪═══════════════╡
│ Madrid ┆ 52500.0       ┆ 2          ┆ 30       ┆ 5000.0        │
│ Lima   ┆ 44500.25      ┆ 2          ┆ 25       ┆ 4999.5        │
│ Tokyo  ┆ 68000.0       ┆ 1          ┆ 35       ┆ 0.0           │
└────────┴───────────────┴────────────┴──────────┴───────────────┘


In [14]:
# CONTEXTO 4: filter — selecciona FILAS donde la expresión es True
resultado_filter = df.filter(pl.col("activo") & (pl.col("salario") > 45000))
print("filter — shape:", resultado_filter.shape)
print(resultado_filter)

filter — shape: (3, 6)
shape: (3, 6)
┌────────┬──────┬─────────┬────────┬───────────────┬────────┐
│ nombre ┆ edad ┆ salario ┆ activo ┆ fecha_ingreso ┆ ciudad │
│ ---    ┆ ---  ┆ ---     ┆ ---    ┆ ---           ┆ ---    │
│ str    ┆ i64  ┆ f64     ┆ bool   ┆ date          ┆ str    │
╞════════╪══════╪═════════╪════════╪═══════════════╪════════╡
│ Alice  ┆ 30   ┆ 50000.0 ┆ true   ┆ 2020-01-15    ┆ Madrid │
│ Carol  ┆ 35   ┆ 68000.0 ┆ true   ┆ 2019-03-20    ┆ Tokyo  │
│ David  ┆ 28   ┆ 55000.0 ┆ true   ┆ 2022-11-10    ┆ Madrid │
└────────┴──────┴─────────┴────────┴───────────────┴────────┘


### Explicación

La diferencia clave entre `select` y `with_columns`:

- **`select`** es destructivo: retorna **solo** lo que pides. Si pides 3 expresiones, obtienes 3 columnas.
- **`with_columns`** es aditivo: retorna **todas** las columnas originales más las nuevas. Si una expresión tiene el mismo nombre que una columna existente, la **reemplaza**.

En `group_by.agg`, observa que puedes poner expresiones complejas dentro de `agg()`. Polars no te limita a funciones predefinidas como `sum` o `mean` — cualquier expresión válida sirve.

> **Referencia:** Sección 17.3 §3 en `03_polars_puro.md` tiene la tabla completa de contextos.

---
## §4: Filter, sort, unique, nulls

Polars maneja filtros, ordenamiento y nulls de forma expresiva. Aquí veremos los patrones más comunes y compararemos el manejo de nulls con pandas.

### Predecir

> **¿Qué pasa con los tipos cuando llenamos nulls en una columna `Int64` con `fill_null(0)`?**
>
> En pandas, si la columna era `float64` (por tener NaN), ¿vuelve a ser `int64`?
> En Polars, ¿cambia el tipo?

In [15]:
# Crear un DataFrame con nulls para trabajar
df_nulls = pl.DataFrame({
    "nombre": ["Alice", "Bob", "Carol", "David", "Eva", "Frank"],
    "depto": ["Ventas", "IT", "Ventas", "IT", "HR", "HR"],
    "salario": [50000, None, 68000, 55000, None, 47000],
    "bonus": [5000, 3000, None, 7000, 2000, None],
})
print(df_nulls)
print(f"\nSchema: {df_nulls.schema}")

shape: (6, 4)
┌────────┬────────┬─────────┬───────┐
│ nombre ┆ depto  ┆ salario ┆ bonus │
│ ---    ┆ ---    ┆ ---     ┆ ---   │
│ str    ┆ str    ┆ i64     ┆ i64   │
╞════════╪════════╪═════════╪═══════╡
│ Alice  ┆ Ventas ┆ 50000   ┆ 5000  │
│ Bob    ┆ IT     ┆ null    ┆ 3000  │
│ Carol  ┆ Ventas ┆ 68000   ┆ null  │
│ David  ┆ IT     ┆ 55000   ┆ 7000  │
│ Eva    ┆ HR     ┆ null    ┆ 2000  │
│ Frank  ┆ HR     ┆ 47000   ┆ null  │
└────────┴────────┴─────────┴───────┘

Schema: Schema([('nombre', String), ('depto', String), ('salario', Int64), ('bonus', Int64)])


In [16]:
# Filtros combinados con & (AND) y | (OR)
# IMPORTANTE: los paréntesis son obligatorios por precedencia de operadores
print("Filtro AND:")
print(df_nulls.filter(
    (pl.col("depto") == "IT") & (pl.col("salario").is_not_null())
))
print()

print("Filtro OR:")
print(df_nulls.filter(
    (pl.col("depto") == "Ventas") | (pl.col("depto") == "HR")
))
print()

# is_in — forma más limpia para múltiples valores
print("Filtro is_in:")
print(df_nulls.filter(pl.col("depto").is_in(["Ventas", "HR"])))

Filtro AND:
shape: (1, 4)
┌────────┬───────┬─────────┬───────┐
│ nombre ┆ depto ┆ salario ┆ bonus │
│ ---    ┆ ---   ┆ ---     ┆ ---   │
│ str    ┆ str   ┆ i64     ┆ i64   │
╞════════╪═══════╪═════════╪═══════╡
│ David  ┆ IT    ┆ 55000   ┆ 7000  │
└────────┴───────┴─────────┴───────┘

Filtro OR:
shape: (4, 4)
┌────────┬────────┬─────────┬───────┐
│ nombre ┆ depto  ┆ salario ┆ bonus │
│ ---    ┆ ---    ┆ ---     ┆ ---   │
│ str    ┆ str    ┆ i64     ┆ i64   │
╞════════╪════════╪═════════╪═══════╡
│ Alice  ┆ Ventas ┆ 50000   ┆ 5000  │
│ Carol  ┆ Ventas ┆ 68000   ┆ null  │
│ Eva    ┆ HR     ┆ null    ┆ 2000  │
│ Frank  ┆ HR     ┆ 47000   ┆ null  │
└────────┴────────┴─────────┴───────┘

Filtro is_in:
shape: (4, 4)
┌────────┬────────┬─────────┬───────┐
│ nombre ┆ depto  ┆ salario ┆ bonus │
│ ---    ┆ ---    ┆ ---     ┆ ---   │
│ str    ┆ str    ┆ i64     ┆ i64   │
╞════════╪════════╪═════════╪═══════╡
│ Alice  ┆ Ventas ┆ 50000   ┆ 5000  │
│ Carol  ┆ Ventas ┆ 68000   ┆ null  │
│ Eva    ┆ HR 

In [17]:
# Manejo de nulls

# 1. fill_null con valor constante
print("fill_null(0):")
filled = df_nulls.with_columns(pl.col("salario").fill_null(0))
print(filled)
print(f"Tipo de salario después de fill_null: {filled['salario'].dtype}")  # sigue siendo Int64
print()

fill_null(0):
shape: (6, 4)
┌────────┬────────┬─────────┬───────┐
│ nombre ┆ depto  ┆ salario ┆ bonus │
│ ---    ┆ ---    ┆ ---     ┆ ---   │
│ str    ┆ str    ┆ i64     ┆ i64   │
╞════════╪════════╪═════════╪═══════╡
│ Alice  ┆ Ventas ┆ 50000   ┆ 5000  │
│ Bob    ┆ IT     ┆ 0       ┆ 3000  │
│ Carol  ┆ Ventas ┆ 68000   ┆ null  │
│ David  ┆ IT     ┆ 55000   ┆ 7000  │
│ Eva    ┆ HR     ┆ 0       ┆ 2000  │
│ Frank  ┆ HR     ┆ 47000   ┆ null  │
└────────┴────────┴─────────┴───────┘
Tipo de salario después de fill_null: Int64



In [18]:
# 2. fill_null con estrategia (forward fill)
print("fill_null(strategy='forward'):")
print(df_nulls.with_columns(
    pl.col("salario").fill_null(strategy="forward")
))
print()

# 3. coalesce — toma el primer valor no-null de varias columnas
print("coalesce(salario, bonus):")
print(df_nulls.with_columns(
    pl.coalesce("salario", "bonus").alias("salario_o_bonus")
))
print()

# 4. drop_nulls — eliminar filas con nulls
print("drop_nulls():")
print(df_nulls.drop_nulls())  # elimina filas con ANY null
print()

print("drop_nulls(subset=['salario']):")
print(df_nulls.drop_nulls(subset=["salario"]))  # solo mira columna salario

fill_null(strategy='forward'):
shape: (6, 4)
┌────────┬────────┬─────────┬───────┐
│ nombre ┆ depto  ┆ salario ┆ bonus │
│ ---    ┆ ---    ┆ ---     ┆ ---   │
│ str    ┆ str    ┆ i64     ┆ i64   │
╞════════╪════════╪═════════╪═══════╡
│ Alice  ┆ Ventas ┆ 50000   ┆ 5000  │
│ Bob    ┆ IT     ┆ 50000   ┆ 3000  │
│ Carol  ┆ Ventas ┆ 68000   ┆ null  │
│ David  ┆ IT     ┆ 55000   ┆ 7000  │
│ Eva    ┆ HR     ┆ 55000   ┆ 2000  │
│ Frank  ┆ HR     ┆ 47000   ┆ null  │
└────────┴────────┴─────────┴───────┘

coalesce(salario, bonus):
shape: (6, 5)
┌────────┬────────┬─────────┬───────┬─────────────────┐
│ nombre ┆ depto  ┆ salario ┆ bonus ┆ salario_o_bonus │
│ ---    ┆ ---    ┆ ---     ┆ ---   ┆ ---             │
│ str    ┆ str    ┆ i64     ┆ i64   ┆ i64             │
╞════════╪════════╪═════════╪═══════╪═════════════════╡
│ Alice  ┆ Ventas ┆ 50000   ┆ 5000  ┆ 50000           │
│ Bob    ┆ IT     ┆ null    ┆ 3000  ┆ 3000            │
│ Carol  ┆ Ventas ┆ 68000   ┆ null  ┆ 68000           │
│ David  ┆

In [19]:
# Demostración: Arrow nulls vs pandas NaN — preservación de tipo

# En Polars, el tipo NO cambia por tener nulls
s_polars = pl.Series("x", [1, 2, None, 4])
print(f"Polars tipo: {s_polars.dtype}")  # Int64 — se preserva
print(f"Polars valores: {s_polars.to_list()}")

# Convertir a pandas para ver la diferencia
import pandas as pd

# Crear la misma serie directamente en pandas
s_pandas = pd.Series([1, 2, None, 4])
print(f"\nPandas tipo: {s_pandas.dtype}")  # float64 — el int se perdió
print(f"Pandas valores: {s_pandas.values}")  # [1. 2. nan 4.]

# La diferencia: en pandas, un null fuerza la conversión de int a float.
# En Polars, el tipo se mantiene Int64 gracias al validity bitmap de Arrow.


Polars tipo: Int64
Polars valores: [1, 2, None, 4]



Pandas tipo: float64
Pandas valores: [ 1.  2. nan  4.]


### Explicación

**Respuesta a la predicción:** En Polars, `fill_null(0)` en una columna `Int64` **mantiene el tipo `Int64`**. No hay promoción de tipos porque los nulls nunca causaron un cambio de tipo en primer lugar.

En pandas clásico (sin ArrowDtype), al tener un null en una columna entera:
1. La columna se promueve a `float64` (para acomodar `NaN`)
2. Al llenar el NaN con `fillna(0)`, la columna **sigue siendo `float64`** — ya no vuelve a `int64` automáticamente
3. Tienes que hacer `.astype(int)` manualmente

En Polars, nada de esto pasa. Los nulls son ciudadanos de primera clase en todos los tipos.

> **Referencia:** Sección 17.3 §4 en `03_polars_puro.md` cubre filtros, sort, unique y manejo de nulls.

---
## §5: UDFs — expresión nativa vs map_batches vs map_elements

**Esta es la sección más importante del notebook.**

Polars ejecuta sus operaciones nativas en Rust, con paralelismo automático y sin el GIL de Python. Cuando usas una función Python (UDF), rompes este pipeline. La pregunta es: **¿cuánto cuesta?**

Tres formas de hacer lo mismo:

| Método | Recibe | Llamadas Python | Velocidad esperada |
|---|---|---|---|
| Expresión nativa | — | 0 | Línea base (Rust) |
| `map_batches` | Series completa | 1 por columna | ~2-5x más lento |
| `map_elements` | Escalar (fila) | 1 por fila | ~50-200x más lento |

### Predecir

> **¿Cuántas veces más lento será `map_elements` vs expresión nativa para 1 millón de filas?**
>
> Pista: `map_elements` hace una llamada Python **por cada fila**. Python ejecuta ~50M operaciones/segundo. Una operación aritmética en Rust sobre un array toma nanosegundos por elemento.

Vamos a medir.

In [20]:
# Crear un DataFrame de 1 millón de filas con temperaturas en Celsius
random.seed(42)
n = 1_000_000
temps = [random.uniform(-20, 45) for _ in range(n)]  # temperaturas aleatorias
df_temp = pl.DataFrame({"temp_c": temps})

print(f"Shape: {df_temp.shape}")
print(df_temp.head())

Shape: (1000000, 1)
shape: (5, 1)
┌────────────┐
│ temp_c     │
│ ---        │
│ f64        │
╞════════════╡
│ 21.562742  │
│ -18.374301 │
│ -2.123094  │
│ -5.491302  │
│ 27.870629  │
└────────────┘


In [21]:
# Tarea: convertir Celsius a Fahrenheit: F = C * 9/5 + 32

# ═══════════════════════════════════════════
# MÉTODO 1: Expresión nativa (Rust puro)
# ═══════════════════════════════════════════
start = time.perf_counter()
r1 = df_temp.with_columns(
    (pl.col("temp_c") * 9 / 5 + 32).alias("temp_f")
)
t_nativa = time.perf_counter() - start
print(f"Expresión nativa: {t_nativa*1000:.2f} ms")

# ═══════════════════════════════════════════
# MÉTODO 2: map_batches (una llamada Python)
# Recibe la Serie completa, opera sobre ella
# ═══════════════════════════════════════════
start = time.perf_counter()
r2 = df_temp.with_columns(
    pl.col("temp_c").map_batches(
        lambda s: s * 9 / 5 + 32  # s es una Series de Polars
    ).alias("temp_f")
)
t_batches = time.perf_counter() - start
print(f"map_batches:      {t_batches*1000:.2f} ms")

# ═══════════════════════════════════════════
# MÉTODO 3: map_elements (una llamada POR FILA)
# Recibe un escalar, retorna un escalar
# ═══════════════════════════════════════════
start = time.perf_counter()
r3 = df_temp.with_columns(
    pl.col("temp_c").map_elements(
        lambda x: x * 9 / 5 + 32,  # x es un float individual
        return_dtype=pl.Float64       # OBLIGATORIO: Polars no infiere el tipo
    ).alias("temp_f")
)
t_elements = time.perf_counter() - start
print(f"map_elements:     {t_elements*1000:.2f} ms")

Expresión nativa: 11.77 ms
map_batches:      7.47 ms


/tmp/ipykernel_1932927/299456928.py:32: PolarsInefficientMapWarning: 
Expr.map_elements is significantly slower than the native expressions API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - pl.col("temp_c").map_elements(lambda x: ...)
with this one instead:
  + ((pl.col("temp_c") * 9) / 5) + 32

  pl.col("temp_c").map_elements(


map_elements:     422.55 ms


In [22]:
# Tabla comparativa de rendimiento
print("\n" + "="*55)
print(f"{'Método':<20} {'Tiempo (ms)':>12} {'Ratio':>10}")
print("="*55)
print(f"{'Expresión nativa':<20} {t_nativa*1000:>12.2f} {1.0:>10.1f}x")
print(f"{'map_batches':<20} {t_batches*1000:>12.2f} {t_batches/t_nativa:>10.1f}x")
print(f"{'map_elements':<20} {t_elements*1000:>12.2f} {t_elements/t_nativa:>10.1f}x")
print("="*55)
print(f"\nmap_elements es ~{t_elements/t_nativa:.0f}x más lento que la expresión nativa.")
print(f"map_batches es ~{t_batches/t_nativa:.0f}x más lento que la expresión nativa.")


Método                Tiempo (ms)      Ratio
Expresión nativa            11.77        1.0x
map_batches                  7.47        0.6x
map_elements               422.55       35.9x

map_elements es ~36x más lento que la expresión nativa.
map_batches es ~1x más lento que la expresión nativa.


In [23]:
# Verificar que los tres métodos dan el mismo resultado
assert r1["temp_f"].equals(r2["temp_f"]), "map_batches difiere!"
# map_elements puede tener pequeñas diferencias de punto flotante
diff = (r1["temp_f"] - r3["temp_f"]).abs().max()
print(f"Diferencia máxima entre nativa y map_elements: {diff}")
print("Los tres métodos producen (esencialmente) el mismo resultado.")

Diferencia máxima entre nativa y map_elements: 1.4210854715202004e-14
Los tres métodos producen (esencialmente) el mismo resultado.


### Explicación

**¿Por qué cada método es más lento?**

**Expresión nativa** (la más rápida):
- Ejecuta todo en Rust, sin pasar por Python
- Opera sobre arrays contiguos en memoria (cache-friendly)
- Usa SIMD (instrucciones vectorizadas del CPU) cuando es posible
- Puede paralelizar automáticamente entre cores

**`map_batches`** (~2-5x más lento):
- Hace **una sola llamada** a Python por columna
- Recibe la Serie completa, así que las operaciones sobre la Serie siguen siendo vectorizadas
- El overhead principal es la transición Rust → Python → Rust (una sola vez)
- Pierde la capacidad de Polars para optimizar/fusionar operaciones

**`map_elements`** (~50-200x más lento):
- Hace **una llamada Python por cada fila** (1 millón de llamadas)
- Cada llamada tiene overhead: crear objeto Python, ejecutar el bytecode, convertir de vuelta
- No hay vectorización, no hay SIMD, no hay paralelismo
- Es esencialmente un loop de Python — lo peor que le puedes pedir a Polars

**Regla de oro:** Siempre intenta expresar tu lógica con expresiones nativas. Solo usa `map_batches` si necesitas una función que opera sobre la Serie completa (ej: scipy). Solo usa `map_elements` como último recurso (ej: llamar a una API por fila).

> **Referencia:** Sección 17.3 §5 en `03_polars_puro.md` tiene el árbol de decisión completo para elegir el método correcto.

---
## §6: Columnas List

Una de las diferencias más importantes entre Polars y pandas: Polars tiene un tipo `List` nativo. Cada celda puede contener una lista de valores del mismo tipo. Esto permite operaciones que en pandas requieren `apply` con funciones Python — aquí se hacen en Rust.

### Predecir

> **¿Qué pasa cuando haces `group_by.agg(pl.col('x'))` sin función de agregación?**
>
> En pandas, `groupby().agg()` requiere una función como `sum`, `mean`, etc.
> En Polars, ¿qué pasa si no pones ninguna función?

In [24]:
# Crear datos para agrupar
df_list = pl.DataFrame({
    "grupo": ["A", "A", "A", "B", "B", "B", "B"],
    "valor": [10, 20, 30, 40, 50, 60, 70],
    "peso":  [1.0, 1.5, 2.0, 0.5, 1.0, 1.5, 2.0],
})

# group_by + agg SIN función de agregación → crea columnas List
agrupado = df_list.group_by("grupo", maintain_order=True).agg(
    pl.col("valor"),  # sin .sum(), .mean(), etc.
    pl.col("peso"),
)

print(agrupado)
print(f"\nTipo de 'valor': {agrupado['valor'].dtype}")  # list[i64]

shape: (2, 3)
┌───────┬────────────────┬───────────────────┐
│ grupo ┆ valor          ┆ peso              │
│ ---   ┆ ---            ┆ ---               │
│ str   ┆ list[i64]      ┆ list[f64]         │
╞═══════╪════════════════╪═══════════════════╡
│ A     ┆ [10, 20, 30]   ┆ [1.0, 1.5, 2.0]   │
│ B     ┆ [40, 50, … 70] ┆ [0.5, 1.0, … 2.0] │
└───────┴────────────────┴───────────────────┘

Tipo de 'valor': List(Int64)


In [25]:
# Operaciones sobre columnas List — todo se ejecuta en Rust
print(agrupado.with_columns(
    pl.col("valor").list.len().alias("n"),            # longitud de cada lista
    pl.col("valor").list.sum().alias("total"),         # suma de cada lista
    pl.col("valor").list.mean().alias("promedio"),     # media de cada lista
    pl.col("valor").list.get(0).alias("primero"),      # primer elemento
    pl.col("valor").list.get(-1).alias("ultimo"),      # último elemento
    pl.col("valor").list.contains(20).alias("tiene_20"),  # ¿contiene 20?
))

shape: (2, 9)
┌───────┬────────────────┬───────────────────┬─────┬───┬──────────┬─────────┬────────┬──────────┐
│ grupo ┆ valor          ┆ peso              ┆ n   ┆ … ┆ promedio ┆ primero ┆ ultimo ┆ tiene_20 │
│ ---   ┆ ---            ┆ ---               ┆ --- ┆   ┆ ---      ┆ ---     ┆ ---    ┆ ---      │
│ str   ┆ list[i64]      ┆ list[f64]         ┆ u32 ┆   ┆ f64      ┆ i64     ┆ i64    ┆ bool     │
╞═══════╪════════════════╪═══════════════════╪═════╪═══╪══════════╪═════════╪════════╪══════════╡
│ A     ┆ [10, 20, 30]   ┆ [1.0, 1.5, 2.0]   ┆ 3   ┆ … ┆ 20.0     ┆ 10      ┆ 30     ┆ true     │
│ B     ┆ [40, 50, … 70] ┆ [0.5, 1.0, … 2.0] ┆ 4   ┆ … ┆ 55.0     ┆ 40      ┆ 70     ┆ false    │
└───────┴────────────────┴───────────────────┴─────┴───┴──────────┴─────────┴────────┴──────────┘


In [26]:
# .list.eval() — la operación más poderosa
# Ejecuta una expresión DENTRO de cada lista
# pl.element() es como pl.col() pero dentro del contexto de una lista

resultado_eval = agrupado.with_columns(
    # Normalizar cada lista: restar la media de ESA lista
    pl.col("valor").list.eval(
        pl.element() - pl.element().mean()   # cada lista se normaliza independientemente
    ).alias("normalizado"),

    # Diferencias consecutivas dentro de cada lista
    pl.col("valor").list.eval(
        pl.element().diff()                  # diff() calcula x[i] - x[i-1]
    ).alias("diffs"),

    # Ordenar cada lista en orden descendente
    pl.col("valor").list.eval(
        pl.element().sort(descending=True)
    ).alias("ordenado_desc"),
)

print(resultado_eval)

shape: (2, 6)
┌───────┬────────────────┬──────────────────┬──────────────────┬──────────────────┬────────────────┐
│ grupo ┆ valor          ┆ peso             ┆ normalizado      ┆ diffs            ┆ ordenado_desc  │
│ ---   ┆ ---            ┆ ---              ┆ ---              ┆ ---              ┆ ---            │
│ str   ┆ list[i64]      ┆ list[f64]        ┆ list[f64]        ┆ list[i64]        ┆ list[i64]      │
╞═══════╪════════════════╪══════════════════╪══════════════════╪══════════════════╪════════════════╡
│ A     ┆ [10, 20, 30]   ┆ [1.0, 1.5, 2.0]  ┆ [-10.0, 0.0,     ┆ [null, 10, 10]   ┆ [30, 20, 10]   │
│       ┆                ┆                  ┆ 10.0]            ┆                  ┆                │
│ B     ┆ [40, 50, … 70] ┆ [0.5, 1.0, …     ┆ [-15.0, -5.0, …  ┆ [null, 10, … 10] ┆ [70, 60, … 40] │
│       ┆                ┆ 2.0]             ┆ 15.0]            ┆                  ┆                │
└───────┴────────────────┴──────────────────┴──────────────────┴─────────────

In [27]:
# Explode e implode — round-trip

# explode: deshacer listas → una fila por elemento
explotado = agrupado.explode("valor", "peso")
print("Después de explode:")
print(explotado)
print()

# implode: de vuelta a listas (reagrupar)
reimplodido = explotado.group_by("grupo", maintain_order=True).agg(
    pl.col("valor"),
    pl.col("peso"),
)
print("Después de implode (group_by + agg):")
print(reimplodido)
print()

# Verificar el round-trip
print(f"¿Round-trip preserva datos? {agrupado.equals(reimplodido)}")

Después de explode:
shape: (7, 3)
┌───────┬───────┬──────┐
│ grupo ┆ valor ┆ peso │
│ ---   ┆ ---   ┆ ---  │
│ str   ┆ i64   ┆ f64  │
╞═══════╪═══════╪══════╡
│ A     ┆ 10    ┆ 1.0  │
│ A     ┆ 20    ┆ 1.5  │
│ A     ┆ 30    ┆ 2.0  │
│ B     ┆ 40    ┆ 0.5  │
│ B     ┆ 50    ┆ 1.0  │
│ B     ┆ 60    ┆ 1.5  │
│ B     ┆ 70    ┆ 2.0  │
└───────┴───────┴──────┘

Después de implode (group_by + agg):
shape: (2, 3)
┌───────┬────────────────┬───────────────────┐
│ grupo ┆ valor          ┆ peso              │
│ ---   ┆ ---            ┆ ---               │
│ str   ┆ list[i64]      ┆ list[f64]         │
╞═══════╪════════════════╪═══════════════════╡
│ A     ┆ [10, 20, 30]   ┆ [1.0, 1.5, 2.0]   │
│ B     ┆ [40, 50, … 70] ┆ [0.5, 1.0, … 2.0] │
└───────┴────────────────┴───────────────────┘

¿Round-trip preserva datos? True


### Explicación

**Respuesta a la predicción:** Cuando haces `group_by.agg(pl.col('x'))` sin función de agregación, Polars **recolecta todos los valores del grupo en una lista**. El resultado es una columna de tipo `list[tipo_original]`.

Esto es profundamente diferente a pandas, donde `groupby().agg()` siempre requiere una función reductora. En Polars, la "no-agregación" es una operación válida y extremadamente útil.

**¿Por qué importa `.list.eval()`?** Porque te permite ejecutar cualquier expresión de Polars *dentro de cada lista*, sin salir de Rust. En pandas, necesitarías `.apply(lambda x: ...)` — una llamada Python por grupo. En Polars, todo se ejecuta de forma vectorizada.

El patrón `group_by → agg → (operaciones sobre listas) → explode` es fundamental para operaciones que necesitan contexto de grupo sin perder rendimiento.

> **Referencia:** Sección 17.3 §6 en `03_polars_puro.md` detalla las columnas List, `.list.eval()` y el round-trip explode/implode.

---
## §7: Struct

Un `Struct` en Polars es un tipo de dato que contiene varios campos nombrados — como un diccionario tipado. Es la forma natural de manejar datos anidados (JSON, respuestas de APIs) sin aplanar todo en columnas separadas.

### ¿Cuándo usar Struct vs columnas separadas?

| Caso | Recomendación |
|---|---|
| Campos que siempre se usan juntos | Struct (ej: coordenadas lat/lon) |
| Datos provenientes de JSON/APIs | Struct (estructura natural) |
| Campos que se filtran/agregan independientemente | Columnas separadas |
| Análisis exploratorio | Columnas separadas (más fácil) |

In [28]:
# Crear structs desde diccionarios
df_struct = pl.DataFrame({
    "nombre": ["Alice", "Bob", "Carol"],
    "coordenadas": [
        {"lat": 40.4168, "lon": -3.7038},   # Madrid
        {"lat": -12.0464, "lon": -77.0428},  # Lima
        {"lat": 35.6762, "lon": 139.6503},   # Tokyo
    ],
})

print(df_struct)
print(f"\nTipo de 'coordenadas': {df_struct['coordenadas'].dtype}")

shape: (3, 2)
┌────────┬─────────────────────┐
│ nombre ┆ coordenadas         │
│ ---    ┆ ---                 │
│ str    ┆ struct[2]           │
╞════════╪═════════════════════╡
│ Alice  ┆ {40.4168,-3.7038}   │
│ Bob    ┆ {-12.0464,-77.0428} │
│ Carol  ┆ {35.6762,139.6503}  │
└────────┴─────────────────────┘

Tipo de 'coordenadas': Struct({'lat': Float64, 'lon': Float64})


In [29]:
# Acceder a campos del struct con .struct.field()
print(df_struct.with_columns(
    pl.col("coordenadas").struct.field("lat").alias("latitud"),
    pl.col("coordenadas").struct.field("lon").alias("longitud"),
))

shape: (3, 4)
┌────────┬─────────────────────┬──────────┬──────────┐
│ nombre ┆ coordenadas         ┆ latitud  ┆ longitud │
│ ---    ┆ ---                 ┆ ---      ┆ ---      │
│ str    ┆ struct[2]           ┆ f64      ┆ f64      │
╞════════╪═════════════════════╪══════════╪══════════╡
│ Alice  ┆ {40.4168,-3.7038}   ┆ 40.4168  ┆ -3.7038  │
│ Bob    ┆ {-12.0464,-77.0428} ┆ -12.0464 ┆ -77.0428 │
│ Carol  ┆ {35.6762,139.6503}  ┆ 35.6762  ┆ 139.6503 │
└────────┴─────────────────────┴──────────┴──────────┘


In [30]:
# Crear structs desde columnas existentes con pl.struct()
df_plano = pl.DataFrame({
    "nombre": ["Alice", "Bob", "Carol"],
    "lat": [40.4168, -12.0464, 35.6762],
    "lon": [-3.7038, -77.0428, 139.6503],
    "ciudad": ["Madrid", "Lima", "Tokyo"],
})

# Empaquetar lat y lon en un struct
df_con_struct = df_plano.with_columns(
    pl.struct("lat", "lon").alias("coordenadas")
)
print(df_con_struct)
print(f"\nTipo: {df_con_struct['coordenadas'].dtype}")

shape: (3, 5)
┌────────┬──────────┬──────────┬────────┬─────────────────────┐
│ nombre ┆ lat      ┆ lon      ┆ ciudad ┆ coordenadas         │
│ ---    ┆ ---      ┆ ---      ┆ ---    ┆ ---                 │
│ str    ┆ f64      ┆ f64      ┆ str    ┆ struct[2]           │
╞════════╪══════════╪══════════╪════════╪═════════════════════╡
│ Alice  ┆ 40.4168  ┆ -3.7038  ┆ Madrid ┆ {40.4168,-3.7038}   │
│ Bob    ┆ -12.0464 ┆ -77.0428 ┆ Lima   ┆ {-12.0464,-77.0428} │
│ Carol  ┆ 35.6762  ┆ 139.6503 ┆ Tokyo  ┆ {35.6762,139.6503}  │
└────────┴──────────┴──────────┴────────┴─────────────────────┘

Tipo: Struct({'lat': Float64, 'lon': Float64})


In [31]:
# Unnest — lo inverso de pl.struct()
# Desempaqueta un struct en columnas separadas
print("Unnest del struct:")
print(df_struct.unnest("coordenadas"))

Unnest del struct:
shape: (3, 3)
┌────────┬──────────┬──────────┐
│ nombre ┆ lat      ┆ lon      │
│ ---    ┆ ---      ┆ ---      │
│ str    ┆ f64      ┆ f64      │
╞════════╪══════════╪══════════╡
│ Alice  ┆ 40.4168  ┆ -3.7038  │
│ Bob    ┆ -12.0464 ┆ -77.0428 │
│ Carol  ┆ 35.6762  ┆ 139.6503 │
└────────┴──────────┴──────────┘


### Explicación

Los `Struct` son la forma en que Polars representa datos anidados de forma tipada. A diferencia de una columna `object` en pandas (que puede contener cualquier cosa), un `Struct` tiene campos con nombres y tipos definidos.

Operaciones clave:
- **`pl.struct("col1", "col2")`** — empaquetar columnas en un struct
- **`.struct.field("nombre")`** — extraer un campo
- **`.unnest("col_struct")`** — desempaquetar el struct en columnas

En la práctica, los Structs aparecen frecuentemente al leer JSON con `pl.read_json()` o `pl.scan_ndjson()`, donde los objetos anidados se representan naturalmente como Structs.

> **Referencia:** Sección 17.3 §7 en `03_polars_puro.md` explica cuándo usar Struct vs columnas separadas.

---
## §8: LazyFrame — scan → transform → .explain() → collect

El modo **lazy** es donde Polars realmente brilla. En lugar de ejecutar cada operación inmediatamente, Polars construye un **plan de ejecución** que optimiza antes de ejecutar.

Las optimizaciones principales son:
- **Projection pushdown:** Solo lee las columnas que realmente necesitas
- **Predicate pushdown:** Aplica filtros lo antes posible (incluso al leer archivos)
- **Common subexpression elimination:** Reutiliza cálculos repetidos

### Predecir

> **Si construyes un pipeline que lee un DataFrame de 10 columnas, filtra por una, transforma otra, y al final selecciona solo 2 columnas — ¿cuántas columnas leerá Polars del DataFrame interno?**
>
> Pista: projection pushdown.

In [32]:
# Crear un DataFrame moderado (100K filas) para medir tiempos
random.seed(42)
n = 100_000

df_lazy_src = pl.DataFrame({
    "id": list(range(n)),
    "categoria": [random.choice(["A", "B", "C", "D"]) for _ in range(n)],
    "region": [random.choice(["Norte", "Sur", "Este", "Oeste"]) for _ in range(n)],
    "monto": [random.uniform(10, 1000) for _ in range(n)],
    "descuento": [random.uniform(0, 0.3) for _ in range(n)],
    "cantidad": [random.randint(1, 100) for _ in range(n)],
    "col_extra_1": [random.random() for _ in range(n)],
    "col_extra_2": [random.random() for _ in range(n)],
    "col_extra_3": [random.random() for _ in range(n)],
    "col_extra_4": [random.random() for _ in range(n)],
})

print(f"Shape: {df_lazy_src.shape}")
print(f"Columnas: {df_lazy_src.columns}")

Shape: (100000, 10)
Columnas: ['id', 'categoria', 'region', 'monto', 'descuento', 'cantidad', 'col_extra_1', 'col_extra_2', 'col_extra_3', 'col_extra_4']


In [33]:
# Paso 1: crear un LazyFrame desde el DataFrame
lf = df_lazy_src.lazy()
print(f"Tipo: {type(lf)}")
# Nada se ha ejecutado todavía

Tipo: <class 'polars.lazyframe.frame.LazyFrame'>


In [34]:
# Paso 2: encadenar transformaciones (nada se ejecuta)
lf_pipeline = (
    lf
    .filter(pl.col("monto") > 100)                       # filtrar filas
    .with_columns(
        (pl.col("monto") * (1 - pl.col("descuento"))).alias("monto_final"),
        (pl.col("monto") * 0.16).alias("iva"),
    )
    .group_by("categoria").agg(
        pl.col("monto_final").sum().alias("total"),
        pl.col("monto_final").mean().alias("promedio"),
        pl.col("id").count().alias("n_ventas"),
    )
    .sort("total", descending=True)
)

print("Pipeline construido. Nada ejecutado todavía.")
print(f"Tipo: {type(lf_pipeline)}")

Pipeline construido. Nada ejecutado todavía.
Tipo: <class 'polars.lazyframe.frame.LazyFrame'>


In [35]:
# Paso 3: inspeccionar el plan — SIN optimizar
# Esto muestra exactamente lo que tú escribiste
print("=" * 50)
print("PLAN SIN OPTIMIZAR:")
print("=" * 50)
print(lf_pipeline.explain(optimized=False))

PLAN SIN OPTIMIZAR:
SORT BY [col("total")]
  AGGREGATE
    [col("monto_final").sum().alias("total"), col("monto_final").mean().alias("promedio"), col("id").count().alias("n_ventas")] BY [col("categoria")]
    FROM
     WITH_COLUMNS:
     [[(col("monto")) * ([(1.0) - (col("descuento"))])].alias("monto_final"), [(col("monto")) * (0.16)].alias("iva")] 
      FILTER [(col("monto")) > (100.0)]
      FROM
        DF ["id", "categoria", "region", "monto", ...]; PROJECT */10 COLUMNS


In [36]:
# Paso 3b: inspeccionar el plan — OPTIMIZADO
# Polars reorganiza operaciones para máximo rendimiento
print("=" * 50)
print("PLAN OPTIMIZADO:")
print("=" * 50)
print(lf_pipeline.explain())

# Compara: ¿qué columnas aparecen en el plan optimizado?
# Las col_extra_* no se mencionan — Polars sabe que no las necesita

PLAN OPTIMIZADO:
SORT BY [col("total")]
  AGGREGATE
    [col("monto_final").sum().alias("total"), col("monto_final").mean().alias("promedio"), col("id").count().alias("n_ventas")] BY [col("categoria")]
    FROM
     WITH_COLUMNS:
     [[(col("monto")) * ([(1.0) - (col("descuento"))])].alias("monto_final")] 
      FILTER [(col("monto")) > (100.0)]
      FROM
        DF ["id", "categoria", "region", "monto", ...]; PROJECT["id", "categoria", "monto", "descuento"] 4/10 COLUMNS


In [37]:
# Paso 4: ejecutar con .collect()
resultado = lf_pipeline.collect()
print(resultado)

shape: (4, 4)
┌───────────┬──────────┬────────────┬──────────┐
│ categoria ┆ total    ┆ promedio   ┆ n_ventas │
│ ---       ┆ ---      ┆ ---        ┆ ---      │
│ str       ┆ f64      ┆ f64        ┆ u32      │
╞═══════════╪══════════╪════════════╪══════════╡
│ D         ┆ 1.0730e7 ┆ 467.262081 ┆ 22963    │
│ A         ┆ 1.0607e7 ┆ 465.52118  ┆ 22786    │
│ B         ┆ 1.0575e7 ┆ 465.561835 ┆ 22715    │
│ C         ┆ 1.0538e7 ┆ 468.892304 ┆ 22475    │
└───────────┴──────────┴────────────┴──────────┘


In [38]:
# Medir: lazy vs eager
# ¿Hay diferencia de tiempo en este caso?

# Eager: ejecutar todo paso a paso
start = time.perf_counter()
for _ in range(10):  # repetir 10 veces para medir mejor
    r_eager = (
        df_lazy_src
        .filter(pl.col("monto") > 100)
        .with_columns(
            (pl.col("monto") * (1 - pl.col("descuento"))).alias("monto_final"),
            (pl.col("monto") * 0.16).alias("iva"),
        )
        .group_by("categoria").agg(
            pl.col("monto_final").sum().alias("total"),
            pl.col("monto_final").mean().alias("promedio"),
            pl.col("id").count().alias("n_ventas"),
        )
        .sort("total", descending=True)
    )
t_eager = (time.perf_counter() - start) / 10

# Lazy: construir plan, optimizar, ejecutar
start = time.perf_counter()
for _ in range(10):
    r_lazy = (
        df_lazy_src.lazy()
        .filter(pl.col("monto") > 100)
        .with_columns(
            (pl.col("monto") * (1 - pl.col("descuento"))).alias("monto_final"),
            (pl.col("monto") * 0.16).alias("iva"),
        )
        .group_by("categoria").agg(
            pl.col("monto_final").sum().alias("total"),
            pl.col("monto_final").mean().alias("promedio"),
            pl.col("id").count().alias("n_ventas"),
        )
        .sort("total", descending=True)
        .collect()
    )
t_lazy = (time.perf_counter() - start) / 10

print(f"Eager: {t_eager*1000:.2f} ms")
print(f"Lazy:  {t_lazy*1000:.2f} ms")
print(f"Ratio: {t_eager/t_lazy:.2f}x")
print()
print("Nota: con datos en memoria y pocas columnas, la diferencia puede ser pequeña.")
print("La ventaja real de lazy aparece al leer archivos (scan_csv/scan_parquet)")
print("donde projection pushdown EVITA leer columnas innecesarias del disco.")

Eager: 16.74 ms
Lazy:  5.85 ms
Ratio: 2.86x

Nota: con datos en memoria y pocas columnas, la diferencia puede ser pequeña.
La ventaja real de lazy aparece al leer archivos (scan_csv/scan_parquet)
donde projection pushdown EVITA leer columnas innecesarias del disco.


### Explicación

**Respuesta a la predicción:** Polars solo leerá las columnas que realmente necesita. En nuestro pipeline:
- `monto` — para el filtro y cálculos
- `descuento` — para calcular `monto_final`
- `categoria` — para el `group_by`
- `id` — para el `count`

Las columnas `region`, `cantidad`, `col_extra_1` a `col_extra_4` **nunca se tocan**. Si estuviéramos leyendo de un archivo Parquet (formato columnar), esas columnas ni siquiera se leerían del disco.

**¿Cuándo importa lazy?**

1. **Archivos grandes:** `scan_parquet`/`scan_csv` + lazy evita cargar datos innecesarios
2. **Pipelines largos:** El optimizador puede fusionar operaciones, eliminar redundancias
3. **Muchas columnas:** Projection pushdown descarta columnas temprano
4. **Filtros selectivos:** Predicate pushdown filtra antes de transformar

Con datos ya en memoria y pocas columnas (como en nuestro ejemplo), la diferencia puede ser mínima. La regla general: **usa lazy por defecto**, eager solo para exploración interactiva rápida.

> **Referencia:** Sección 17.3 §8 en `03_polars_puro.md` tiene el flujo completo lazy y el árbol de decisión lazy vs eager.

---
## Resumen

| Concepto | Punto clave |
|---|---|
| **Series/DataFrame** | Tipos estrictos, nulls nativos (no NaN), sin índice |
| **Expresiones** | Objetos que describen, no ejecutan. Se componen por encadenamiento |
| **Contextos** | `select` (solo esas cols), `with_columns` (todas + nuevas), `group_by.agg`, `filter` |
| **Nulls** | Arrow bitmap — el tipo no cambia por tener nulls |
| **UDFs** | Expresión nativa >> `map_batches` >> `map_elements` |
| **List** | Tipo nativo, `.list.eval()` para operar dentro de listas sin Python |
| **Struct** | Datos anidados tipados, `.struct.field()` para acceder |
| **LazyFrame** | `.lazy()` → transformaciones → `.explain()` → `.collect()` |